# 🚀 Hello Agent OS

> **Your first governed agent in under 5 minutes.**

## Learning Objectives

By the end of this notebook, you will:
1. Install Agent OS
2. Create a basic governed agent
3. Understand kernel vs. user space
4. See policy enforcement in action

---

## Step 1: Install Agent OS

Run this cell to install the package:

In [ ]:
!pip install -e ../.. --quiet

## Step 2: Import and Initialize

The current beginner path uses `StatelessKernel` plus `ExecutionContext`.

- `StatelessKernel` performs deterministic policy checks before execution.
- `ExecutionContext` carries the agent identity and which built-in policies to enforce for this request.
- This notebook uses the built-in `read_only` and `no_pii` policies, so it runs fully offline.

In [1]:
from agent_os.stateless import StatelessKernel, ExecutionContext

# Create a stateless kernel and an execution context.
kernel = StatelessKernel()
ctx = ExecutionContext(agent_id="hello-agent", policies=["read_only"])

print("✅ StatelessKernel initialized")
print(f"   Agent ID    : {ctx.agent_id}")
print(f"   Policies    : {ctx.policies}")

C:\Users\jbao6\AppData\Local\Temp\ipykernel_39772\1271135400.py:1: DeprecationWarning: agent-os-kernel is deprecated. Use agent-governance-toolkit-core instead. See https://github.com/microsoft/agent-governance-toolkit/blob/main/docs/package-consolidation/MIGRATION.md
  from agent_os import KernelSpace


TypeError: KernelSpace.__init__() got an unexpected keyword argument 'policy'

## Step 3: Execute a Safe Action

Instead of registering a function, the current quickstart sends an explicit action plus parameters through the stateless kernel. The kernel checks policy first, then returns an `ExecutionResult`.

In [ ]:
result = await kernel.execute(
    "respond",
    {"message": "Summarize today's news"},
    ctx,
)

print("✅ Safe action executed through the kernel")
print(f"   success : {result.success}")
print(f"   data    : {result.data}")
print(f"   signal  : {result.signal}")

if result.updated_context:
    ctx = result.updated_context

## Step 4: Blocked Actions Under `read_only`

The built-in `read_only` policy blocks write-oriented actions such as `file_write`, `database_write`, and `send_email`. This demonstrates deterministic pre-execution enforcement.

In [ ]:
blocked = await kernel.execute(
    "file_write",
    {"path": "/tmp/secret.txt", "data": "sensitive data"},
    ctx,
)

print(f"success : {blocked.success}")
print(f"signal  : {blocked.signal}")
print(f"error   : {blocked.error}")

if blocked.updated_context:
    ctx = blocked.updated_context

## 🎉 What Just Happened?

You just saw both sides of Agent OS governance:

```
Request -> Policy Check -> Execute -> Result
          (read_only)     (safe)    success=True

Request -> Policy Check -> Block  -> Result
          (read_only)     (write)   success=False, signal=SIGKILL
```

The important idea is that the policy check happens before the risky action is allowed to proceed.

---

## Step 5: Pattern-Based Blocking with `no_pii`

The built-in `no_pii` policy scans the request parameters for sensitive content such as `password`, `ssn`, `credit_card`, and similar terms.

In [ ]:
ctx_pii = ExecutionContext(agent_id="hello-agent", policies=["no_pii"])

blocked_pii = await kernel.execute(
    "respond",
    {"message": "my password is hunter2"},
    ctx_pii,
)

print(f"success : {blocked_pii.success}")
print(f"signal  : {blocked_pii.signal}")
print(f"error   : {blocked_pii.error}")

In [ ]:
# Optional: inspect the accumulated history from the read_only flow.
if ctx.history:
    print("Audit history:")
    for i, entry in enumerate(ctx.history, 1):
        print(f"  {i}. {entry['action']} at {entry['timestamp']} success={entry['success']}")
else:
    print("No history entries recorded yet.")

## Key Concepts

| Concept | Description |
|---------|-------------|
| **`StatelessKernel`** | Deterministic policy gate that evaluates actions before execution |
| **`ExecutionContext`** | Carries agent identity, active policies, and request history |
| **Action** | A named operation like `respond`, `file_write`, or `send_email` |
| **Policy** | Rules about what actions or content are allowed |
| **`ExecutionResult`** | Structured result with `success`, `data`, `error`, and `signal` |
| **`SIGKILL`** | Returned signal when a policy violation blocks an action |

---

## Step 6: Add an LLM (Optional)

For real model-backed agents, use one of the framework adapters such as `LangChainKernel`. The example below is a sketch only; it requires extra packages and provider credentials.

In [ ]:
# Sketch only: replace with your provider and credentials.

# from agent_os.integrations import GovernancePolicy, LangChainKernel
# from langchain_openai import ChatOpenAI
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.output_parsers import StrOutputParser
#
# llm = ChatOpenAI(temperature=0)
# prompt = ChatPromptTemplate.from_template("Answer briefly: {query}")
# chain = prompt | llm | StrOutputParser()
# chain.name = "qa-chain"
#
# policy = GovernancePolicy(
#     blocked_patterns=["password", "secret", "api_key"],
#     max_tool_calls=2,
# )
# lc_kernel = LangChainKernel(policy=policy)
# governed = lc_kernel.wrap(chain)  # Simple tutorial path; middleware preferred in production.
#
# print(governed.invoke({"query": "What is 2+2?"}))

## Next Steps

Continue learning with these notebooks:

| Notebook | Description | Time |
|----------|-------------|------|
| [02-episodic-memory-demo](02-episodic-memory-demo.ipynb) | Agent memory that persists | 15 min |
| [03-time-travel-debugging](03-time-travel-debugging.ipynb) | Replay and debug decisions | 20 min |
| [04-verification](04-verification.ipynb) | Detect hallucinations | 15 min |
| [05-multi-agent-coordination](05-multi-agent-coordination.ipynb) | Trust between agents | 20 min |
| [06-policy-engine](06-policy-engine.ipynb) | Deep dive into policies | 15 min |

---

## Summary

```python
from agent_os.stateless import StatelessKernel, ExecutionContext

kernel = StatelessKernel()
ctx = ExecutionContext(agent_id="my-agent", policies=["read_only"])

result = await kernel.execute(
    "respond",
    {"message": "Hello, Agent OS!"},
    ctx,
)
```

This is the repo-current beginner pattern used elsewhere in the Agent OS notebooks.